**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 37: A2A(Agent-to-Agent) 프로토콜 기반 멀티 에이전트

## 🎯 학습 목표

**MCP(Session 36)**는 "LLM ↔ 도구" 통신 표준이었습니다. 하지만 진짜 복잡한 작업은 보통 여러 **에이전트가 협력**해야 합니다. 예를 들어:
- 여행 계획 에이전트가 항공권 검색 에이전트와 호텔 예약 에이전트에게 작업을 위임
- 코드 리뷰 에이전트가 보안 분석 에이전트의 결과를 받아 종합
- 데이터 분석 에이전트가 시각화 에이전트에게 차트 생성을 요청

이런 **에이전트 간 통신을 표준화**하는 것이 **A2A(Agent-to-Agent) 프로토콜**입니다. Google이 2025년 4월에 발표했고 50+ 파트너(Salesforce, SAP, MongoDB 등)가 참여 중이며, 현재 Linux Foundation 산하 오픈 거버넌스로 이전되었습니다.

### 이번 세션에서 배울 내용
- ✅ A2A의 동기와 MCP/LangGraph Supervisor와의 차이
- ✅ 핵심 개념: **Agent Card / Task / Message / Part / Artifact**
- ✅ Task 생명주기 (submitted → working → completed / failed)
- ✅ **A2A 호환 서버 두 개를 직접 구현** (Planner + Executor)
- ✅ Agent Discovery → Task 위임 → Artifact 수신 end-to-end
- ✅ 공식 `a2a-sdk`와 우리 구현 비교

### 실습 환경
- 🔧 GPU 불필요 (HTTP + LLM API)
- 🔧 OpenAI API 키 (LLM 백엔드용)
- 🔧 FastAPI + httpx (in-process로 두 에이전트 통신 시뮬레이션)

---
## 1️⃣ A2A가 해결하는 문제

### 세 가지 "멀티 에이전트"의 차이

| 방식 | 통신 방법 | 결합도 | 대표 예 |
|------|----------|--------|---------|
| **LangGraph Supervisor** | 같은 프로세스 내 함수 호출 | 강결합 (코드 공유) | Session 38 |
| **MCP (Session 36)** | LLM → 함수형 도구 호출 | 도구는 함수에 가까움 (스테이트리스) | filesystem, github 서버 |
| **A2A** | 에이전트 간 HTTP JSON-RPC | **느슨한 결합** (다른 회사/언어/모델도 가능) | 본 세션 |

### 왜 별도 프로토콜이 필요한가?

에이전트는 도구와 다음과 같이 다릅니다:
1. **장기 실행**: 작업이 수 초~수 시간 걸릴 수 있음 (단순 함수 호출이 아님)
2. **상태 보유**: 진행 중인 Task를 추적해야 함
3. **불투명한 내부**: 상대 에이전트가 어떤 LLM/도구를 쓰는지 알 필요 없음
4. **양방향 대화**: 추가 정보 요청(`input-required`) 등 turn-taking 필요
5. **다양한 산출물**: 텍스트뿐 아니라 파일, 구조화된 데이터(Part)

> 💡 **포지셔닝**: MCP가 "에이전트에게 도구를 제공"하는 프로토콜이라면, A2A는 "**에이전트끼리 작업을 위임/협업**"하는 프로토콜입니다. 둘은 보완 관계로 같이 쓰입니다.

## 2️⃣ A2A 핵심 개념

### Agent Card (`/.well-known/agent.json`)
에이전트의 "명함". 어떤 일을 할 수 있는지(skills), 어떤 인증이 필요한지를 선언합니다. 클라이언트는 URL만 알면 이 카드를 받아 capability discovery를 합니다.

```json
{
  "name": "Math Executor",
  "description": "산술 계산을 수행합니다",
  "url": "http://localhost:9001/",
  "version": "1.0.0",
  "capabilities": { "streaming": false },
  "skills": [
    {"id": "calc", "name": "Calculate", "description": "수식을 계산"}
  ]
}
```

### Task 생명주기
```
          ┌─ canceled
submitted ─▶ working ─▶ input-required ─▶ working ─▶ completed
                    └────────────────────────────▶ failed
```

| 상태 | 의미 |
|------|------|
| `submitted` | 요청은 받았으나 아직 시작 전 |
| `working` | 에이전트가 처리 중 |
| `input-required` | 추가 정보가 필요해 클라이언트에게 되묻는 상태 |
| `completed` | 정상 완료, `artifacts` 필드에 결과 |
| `failed` / `canceled` | 실패 / 취소 |

### Message / Part / Artifact

| 개념 | 역할 |
|------|------|
| **Message** | Task 내 한 차례의 발언. `role`은 "user" 또는 "agent" |
| **Part** | Message의 한 조각. `text` / `file` / `data`(임의 JSON) 세 타입 |
| **Artifact** | Task 결과물. 여러 Part로 구성될 수 있음 (예: 텍스트 + PNG) |

### 표준 메서드 (JSON-RPC 2.0)
| 메서드 | 용도 |
|--------|------|
| `message/send` | Task 생성 또는 기존 Task에 메시지 추가 (동기) |
| `message/stream` | 위와 같으나 SSE로 진행 상황 스트리밍 |
| `tasks/get` | Task 상태/결과 폴링 |
| `tasks/cancel` | Task 취소 |

---
## 3️⃣ 환경 설정

이 노트북에서는 두 개의 A2A 호환 서버를 만들고, 노트북 안에서 HTTP로 통신시킵니다. 실제 포트를 열지 않기 위해 `httpx.ASGITransport`로 in-process 호출을 합니다.

In [ ]:
!pip install -q fastapi httpx openai python-dotenv pydantic

In [ ]:
import os, json, uuid, asyncio
from typing import Optional, Literal
from datetime import datetime, timezone

from fastapi import FastAPI
from pydantic import BaseModel, Field
import httpx
from dotenv import load_dotenv

load_dotenv("../.env")
if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key를 입력하세요: ")

print("✅ 환경 설정 완료")

---
## 4️⃣ A2A 데이터 모델 정의 (Pydantic)

A2A 스펙의 핵심 타입을 Pydantic 모델로 정의합니다. 실제 `a2a-sdk`도 동일한 형태로 정의되어 있습니다.

In [ ]:
# ----- Part 타입들 -----
class TextPart(BaseModel):
    kind: Literal["text"] = "text"
    text: str

class DataPart(BaseModel):
    kind: Literal["data"] = "data"
    data: dict

Part = TextPart | DataPart  # FilePart는 생략 (스펙에는 존재)

# ----- Message -----
class Message(BaseModel):
    role: Literal["user", "agent"]
    parts: list[Part]
    messageId: str = Field(default_factory=lambda: str(uuid.uuid4()))
    taskId: Optional[str] = None

# ----- Artifact -----
class Artifact(BaseModel):
    artifactId: str = Field(default_factory=lambda: str(uuid.uuid4()))
    name: Optional[str] = None
    parts: list[Part]

# ----- Task Status -----
TaskState = Literal["submitted", "working", "input-required", "completed", "failed", "canceled"]

class TaskStatus(BaseModel):
    state: TaskState
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    message: Optional[Message] = None

# ----- Task -----
class Task(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    contextId: str = Field(default_factory=lambda: str(uuid.uuid4()))
    status: TaskStatus
    history: list[Message] = []
    artifacts: list[Artifact] = []

# ----- JSON-RPC 2.0 -----
class JsonRpcRequest(BaseModel):
    jsonrpc: Literal["2.0"] = "2.0"
    id: str | int
    method: str
    params: dict

class JsonRpcResponse(BaseModel):
    jsonrpc: Literal["2.0"] = "2.0"
    id: str | int
    result: Optional[dict] = None
    error: Optional[dict] = None

print("✅ A2A 데이터 모델 정의 완료")

---
## 5️⃣ Executor Agent — 산술 계산 전문 에이전트

첫 번째 에이전트는 **수식을 받아 계산해서 결과를 돌려주는** 단순 전문가입니다. A2A 표준에 따라:
1. `GET /.well-known/agent.json` → Agent Card 반환
2. `POST /` → JSON-RPC 메서드 처리 (`message/send`, `tasks/get`)

In [ ]:
executor_app = FastAPI(title="Math Executor Agent")
EXECUTOR_TASKS: dict[str, Task] = {}  # 메모리 내 task 저장소

EXECUTOR_CARD = {
    "name": "Math Executor",
    "description": "수학 수식을 계산하는 에이전트",
    "url": "http://math-executor.local/",
    "version": "1.0.0",
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text", "data"],
    "capabilities": {"streaming": False, "pushNotifications": False},
    "skills": [
        {
            "id": "calculate",
            "name": "Calculate",
            "description": "파이썬 수식 문자열을 받아 계산 결과를 반환",
            "tags": ["math", "arithmetic"],
            "examples": ["(2 + 3) * 7", "123 ** 2"],
        }
    ],
}

@executor_app.get("/.well-known/agent.json")
async def executor_card():
    return EXECUTOR_CARD

def _extract_text(msg: Message) -> str:
    return " ".join(p.text for p in msg.parts if isinstance(p, TextPart))

def _safe_eval(expr: str) -> float:
    return eval(expr, {"__builtins__": {}}, {})

@executor_app.post("/")
async def executor_rpc(req: JsonRpcRequest):
    if req.method == "message/send":
        message = Message(**req.params["message"])
        expr = _extract_text(message).strip()
        try:
            result = _safe_eval(expr)
            artifact = Artifact(
                name="calc-result",
                parts=[
                    TextPart(text=f"{expr} = {result}"),
                    DataPart(data={"expression": expr, "result": result}),
                ],
            )
            task = Task(
                status=TaskStatus(state="completed"),
                history=[message],
                artifacts=[artifact],
            )
        except Exception as e:
            task = Task(
                status=TaskStatus(
                    state="failed",
                    message=Message(role="agent", parts=[TextPart(text=f"계산 실패: {e}")]),
                ),
                history=[message],
            )
        EXECUTOR_TASKS[task.id] = task
        return JsonRpcResponse(id=req.id, result=task.model_dump()).model_dump()
    
    if req.method == "tasks/get":
        task_id = req.params["id"]
        task = EXECUTOR_TASKS.get(task_id)
        if not task:
            return JsonRpcResponse(id=req.id, error={"code": -32001, "message": "Task not found"}).model_dump()
        return JsonRpcResponse(id=req.id, result=task.model_dump()).model_dump()
    
    return JsonRpcResponse(id=req.id, error={"code": -32601, "message": f"Method not found: {req.method}"}).model_dump()

print("✅ Executor Agent 정의 완료")
print(f"   - Agent Card: GET /.well-known/agent.json")
print(f"   - JSON-RPC:   POST / (message/send, tasks/get)")

---
## 6️⃣ 미니멀 A2A 클라이언트

Agent Card를 가져오고, JSON-RPC로 메시지를 보내는 헬퍼를 만듭니다.

In [ ]:
class A2AClient:
    """임의의 A2A 호환 에이전트에 연결하는 클라이언트.
    실제 환경에서는 base_url="https://example.com" 으로 외부 호출.
    여기서는 in-process FastAPI 앱을 transport에 직접 꽂아 노트북에서 동작시킴.
    """
    def __init__(self, app: FastAPI, base_url: str = "http://agent.local"):
        self._client = httpx.AsyncClient(
            transport=httpx.ASGITransport(app=app),
            base_url=base_url,
        )
        self.base_url = base_url
        self._card: Optional[dict] = None
    
    async def fetch_card(self) -> dict:
        r = await self._client.get("/.well-known/agent.json")
        r.raise_for_status()
        self._card = r.json()
        return self._card
    
    async def send_message(self, text: str, task_id: Optional[str] = None) -> dict:
        msg = Message(role="user", parts=[TextPart(text=text)], taskId=task_id)
        rpc = JsonRpcRequest(id=str(uuid.uuid4()), method="message/send", params={"message": msg.model_dump()})
        r = await self._client.post("/", json=rpc.model_dump())
        r.raise_for_status()
        return r.json()
    
    async def get_task(self, task_id: str) -> dict:
        rpc = JsonRpcRequest(id=str(uuid.uuid4()), method="tasks/get", params={"id": task_id})
        r = await self._client.post("/", json=rpc.model_dump())
        r.raise_for_status()
        return r.json()
    
    async def close(self):
        await self._client.aclose()

# Executor에 연결해서 동작 확인
executor_client = A2AClient(executor_app, base_url="http://math-executor.local")

card = await executor_client.fetch_card()
print("📛 Agent Card:")
print(json.dumps(card, indent=2, ensure_ascii=False))

In [ ]:
# 단일 Task 송수신
resp = await executor_client.send_message("(2 + 3) * 7")
task = resp["result"]

print(f"📋 Task ID: {task['id']}")
print(f"📋 Status:  {task['status']['state']}")
print(f"\n📦 Artifacts:")
for art in task["artifacts"]:
    for part in art["parts"]:
        print(f"   [{part['kind']}] {part.get('text') or part.get('data')}")

# 실패 케이스
print("\n--- 실패 시나리오 ---")
bad = await executor_client.send_message("이건 수식이 아닙니다")
print(f"Status: {bad['result']['status']['state']}")
print(f"Reason: {bad['result']['status']['message']['parts'][0]['text']}")

---
## 7️⃣ Planner Agent — Executor에게 작업을 위임

두 번째 에이전트는 **사용자의 복잡한 요청을 받아 LLM으로 계획을 세우고, 산술 부분은 Executor에게 위임**합니다. 핵심은 Planner도 똑같이 A2A 인터페이스를 노출하므로, 그 위에 또 다른 상위 에이전트가 올 수 있다는 점입니다.

```
사용자 ──▶ Planner Agent ──A2A 위임──▶ Executor Agent
                │                          │
                │◀─── artifacts ────────── │
                │
         최종 응답 작성
                ▼
             사용자
```

In [ ]:
from openai import AsyncOpenAI

planner_app = FastAPI(title="Planner Agent")
PLANNER_TASKS: dict[str, Task] = {}
openai_async = AsyncOpenAI()

PLANNER_CARD = {
    "name": "Math Word-Problem Planner",
    "description": "자연어 수학 문제를 분석하여 수식으로 변환 후 Math Executor에게 위임",
    "url": "http://planner.local/",
    "version": "1.0.0",
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text"],
    "capabilities": {"streaming": False, "pushNotifications": False},
    "skills": [
        {
            "id": "solve-word-problem",
            "name": "Solve Word Problem",
            "description": "자연어로 된 수학 문제를 풀이",
            "tags": ["math", "reasoning"],
            "examples": ["사과 3개에 한 개당 1500원씩이면 총 얼마?"],
        }
    ],
}

@planner_app.get("/.well-known/agent.json")
async def planner_card():
    return PLANNER_CARD

async def extract_expression(question: str) -> str:
    """LLM이 자연어 문제 → 파이썬 수식으로 변환"""
    resp = await openai_async.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "사용자의 한국어 수학 문제를 분석해서, "
                "파이썬 eval로 계산할 수 있는 단일 수식 한 줄만 출력하세요. "
                "설명, 따옴표, ``` 등은 절대 붙이지 마세요."
            )},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip().strip("`").strip()

async def narrate_answer(question: str, expression: str, result) -> str:
    """계산 결과를 자연스러운 한국어 답변으로 마무리"""
    resp = await openai_async.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "수학 문제와 계산 결과를 받아 친절한 한국어 풀이 답변을 1~2 문장으로 작성하세요."},
            {"role": "user", "content": f"문제: {question}\n수식: {expression}\n결과: {result}"},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

@planner_app.post("/")
async def planner_rpc(req: JsonRpcRequest):
    if req.method != "message/send":
        return JsonRpcResponse(id=req.id, error={"code": -32601, "message": "Unsupported"}).model_dump()
    
    message = Message(**req.params["message"])
    question = _extract_text(message)
    
    # 1) submitted → working
    task = Task(status=TaskStatus(state="working"), history=[message])
    PLANNER_TASKS[task.id] = task
    
    try:
        # 2) LLM으로 수식 추출
        expression = await extract_expression(question)
        
        # 3) Executor에게 A2A 위임 (다른 에이전트 호출!)
        exec_resp = await executor_client.send_message(expression)
        exec_task = exec_resp["result"]
        
        if exec_task["status"]["state"] != "completed":
            raise RuntimeError(f"Executor 실패: {exec_task['status']}")
        
        result = exec_task["artifacts"][0]["parts"][1]["data"]["result"]
        
        # 4) LLM으로 최종 답변 생성
        answer = await narrate_answer(question, expression, result)
        
        # 5) Task 완료 + Artifact 부착
        task.status = TaskStatus(state="completed")
        task.artifacts = [
            Artifact(
                name="answer",
                parts=[
                    TextPart(text=answer),
                    DataPart(data={
                        "question": question,
                        "expression": expression,
                        "result": result,
                        "delegated_task_id": exec_task["id"],
                    }),
                ],
            )
        ]
    except Exception as e:
        task.status = TaskStatus(
            state="failed",
            message=Message(role="agent", parts=[TextPart(text=str(e))]),
        )
    
    PLANNER_TASKS[task.id] = task
    return JsonRpcResponse(id=req.id, result=task.model_dump()).model_dump()

planner_client = A2AClient(planner_app, base_url="http://planner.local")
print("✅ Planner Agent 정의 완료")

---
## 8️⃣ End-to-End 시연 — 두 에이전트 협업

이제 "외부 사용자"가 Planner에게만 말을 걸고, Planner가 내부적으로 Executor에게 A2A로 위임하는 전체 흐름을 확인합니다.

In [ ]:
# 1) Planner의 Agent Card 확인 (Discovery)
planner_card_dict = await planner_client.fetch_card()
print("📛 Planner Agent Card:")
print(f"   name:  {planner_card_dict['name']}")
print(f"   skill: {planner_card_dict['skills'][0]['id']}")

# 2) 자연어 문제 전송
questions = [
    "사과 3개에 한 개당 1500원이고, 봉투값 500원이 추가될 때 총 얼마인가요?",
    "반지름이 7인 원의 넓이는 얼마인가요? (π=3.14159)",
    "2의 10승에서 1024를 뺀 값은?",
]

for q in questions:
    print("\n" + "=" * 70)
    print(f"👤 사용자 → Planner: {q}")
    
    resp = await planner_client.send_message(q)
    task = resp["result"]
    
    print(f"📋 Task {task['id'][:8]}... → {task['status']['state']}")
    
    if task["status"]["state"] == "completed":
        ans_part, meta_part = task["artifacts"][0]["parts"]
        print(f"🤖 답변: {ans_part['text']}")
        print(f"📊 메타: 수식={meta_part['data']['expression']!r}, "
              f"위임 Task={meta_part['data']['delegated_task_id'][:8]}...")
    else:
        print(f"❌ 실패: {task['status'].get('message')}")

### Task 폴링 (`tasks/get`)

장기 실행 Task의 경우 클라이언트는 `tasks/get`으로 상태를 추적합니다. 우리 구현에서는 `message/send`가 동기로 끝나지만, 진짜 비동기 워크플로우에서는 같은 ID로 여러 번 polling 합니다.

In [ ]:
# 직전 Task ID를 다시 조회
last_task_id = task["id"]
polled = await planner_client.get_task(last_task_id)
polled_task = polled["result"]

print(f"📋 Polled Task: {polled_task['id']}")
print(f"   state:      {polled_task['status']['state']}")
print(f"   timestamp:  {polled_task['status']['timestamp']}")
print(f"   history:    {len(polled_task['history'])} message(s)")
print(f"   artifacts:  {len(polled_task['artifacts'])}")

# 존재하지 않는 ID 폴링 → 에러 응답
missing = await planner_client.get_task("does-not-exist")
print(f"\n❌ 잘못된 ID: error={missing.get('error')}")

---
## 9️⃣ A2A의 확장 포인트

우리가 구현한 것은 최소 기능입니다. 실전 A2A는 추가로 다음을 다룹니다.

| 기능 | 의미 | 구현 힌트 |
|------|------|----------|
| **Streaming** (`message/stream`) | Task 진행 상황을 SSE로 푸시 | FastAPI의 `EventSourceResponse` |
| **Push Notification** | 에이전트가 클라이언트 콜백 URL로 결과 전송 | `pushNotificationConfig.url` 등록 |
| **input-required** 상태 | 에이전트가 추가 정보 요청 → 사용자 → 같은 task 재개 | `taskId` 유지하며 새 메시지 전송 |
| **인증/권한** | API Key, OAuth2, mTLS | Agent Card의 `securitySchemes` |
| **FilePart** | 이미지/PDF 등 바이너리 산출물 | base64 또는 외부 URI |
| **MCP와 조합** | 각 에이전트 내부는 MCP로 도구 사용 | A2A는 "에이전트 간", MCP는 "에이전트→도구" |

### 공식 SDK
프로덕션에서는 직접 구현 대신 [`a2a-sdk`](https://github.com/a2aproject/a2a-python)를 추천합니다:

```bash
pip install a2a-sdk
```

SDK는 우리가 손으로 짠 Pydantic 모델, JSON-RPC 라우팅, 스트리밍, 인증을 모두 제공합니다. 우리 노트북 구현은 **"내부에서 무슨 일이 일어나는지"**를 이해하기 위한 학습용입니다.

## 🔁 세 가지 멀티 에이전트 방식 비교 (재확인)

| 측면 | LangGraph Supervisor (Session 38) | MCP (Session 36) | A2A (Session 37) |
|------|----------------------------------|------------------|------------------|
| **누가 누구를 부르나** | Supervisor → 에이전트 노드 | LLM → 도구 서버 | 에이전트 → 에이전트 |
| **통신** | 같은 프로세스 내 함수 호출 | stdio / HTTP+SSE | HTTP JSON-RPC |
| **상태 모델** | LangGraph State | 스테이트리스 도구 호출 | Task 생명주기 |
| **상대를 알아야 하나** | 그래프에 직접 등록 | 도구 스키마 자동 발견 | Agent Card 발견 |
| **언어/벤더 독립** | Python 한정 | 다언어 SDK | 완전 독립 (HTTP만 알면 됨) |
| **장기 실행** | 가능 (체크포인팅) | 도구 한정 단발 호출 | **1차 시민** (Task 상태 모델) |
| **적합한 경우** | 한 팀이 만든 빠른 멀티에이전트 | 도구 공유/생태계 | 조직/벤더 경계를 넘는 협업 |

---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 |
|------|------|
| **A2A 동기** | 조직/벤더/언어 경계를 넘어 에이전트끼리 협업하는 표준 |
| **핵심 자료구조** | Agent Card / Task / Message / Part / Artifact |
| **Task 생명주기** | submitted → working → (input-required) → completed / failed |
| **표준 메서드** | `message/send`, `message/stream`, `tasks/get`, `tasks/cancel` |
| **Discovery** | `/.well-known/agent.json`에서 capability 광고 |
| **위임 패턴** | Planner가 자기도 A2A 서버이면서 다른 A2A 서버를 호출 |
| **MCP와의 관계** | MCP=에이전트↔도구, A2A=에이전트↔에이전트 — 함께 사용 |

### 산출물
- 🟢 `executor_app` — 산술 계산 A2A 에이전트
- 🟢 `planner_app` — LLM으로 문제를 해석하고 executor에게 위임하는 상위 에이전트
- 🟢 `A2AClient` — Agent Card 조회 + JSON-RPC 호출 헬퍼

### 다음 노트북
👉 **Session 38** (`38_agent_tech_stack_langgraph.ipynb`): Agent AI 기술 스택과 LangGraph 기반 워크플로우 — 단일 프로세스 안에서의 멀티 에이전트 오케스트레이션을 더 깊이 들여다봅니다.

In [ ]:
await executor_client.close()
await planner_client.close()
print("Session 37: A2A 프로토콜 기반 멀티 에이전트 실습 완료!")
print("\n다음 단계 추천:")
print("  1. Executor를 진짜 uvicorn으로 별 포트에 띄우고 Planner에서 외부 URL로 호출해 보기")
print("  2. message/stream을 SSE로 구현해 진행 상황 스트리밍")
print("  3. 공식 a2a-sdk로 재구현하며 우리 코드와 비교")
print("     → https://github.com/a2aproject/a2a-python")